In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "8"

본 노트북은 YOLO pose 모델에서 사용되는 평가 지표(mAP, P, R)를
ONNX 변환 모델에서도 최대한 동일한 기준으로 재현하기 위해 작성되었다.

- 평가 데이터: validation split
- 평가 기준: OKS 기반 Pose mAP
- 비교 대상:
  - YOLO pt 모델
  - Distilled ONNX 모델

### 목적
- YOLO pose (PyTorch) vs ONNX pose 모델 성능 비교
- 동일한 validation dataset 기준
- 관절 좌표 정확도 및 jitter 평가
- YOLO pose metric(mAP)과 최대한 유사한 평가 파이프라인 구성



In [ ]:
import cv2
import yaml
import numpy as np
import onnxruntime as ort
from pathlib import Path
from tqdm import tqdm

In [ ]:
data_yaml = Path("../configs/data.yaml").resolve()

with open(data_yaml) as f:
    data = yaml.safe_load(f)

root = data_yaml.parent.parent.parent  # /home/j-i14a203/workspace

val_img_dir = (root / 'datasets'/ data["val"]).resolve()
val_label_dir = Path(str(val_img_dir).replace("images", "labels"))

print("val_img_dir:", val_img_dir)
print("exists:", val_img_dir.exists())

In [ ]:
val_imgs = sorted(val_img_dir.glob("*.jpg"))
len(val_imgs)

In [ ]:
def load_gt(label_path, kpt_num=21):
    if not label_path.exists():
        return None

    with open(label_path) as f:
        line = f.readline().strip().split()

    kpts = np.array(line[5:], dtype=np.float32).reshape(kpt_num, 3)
    return kpts


In [ ]:
onnx_path = "../weights/export/yolo11n-pose.onnx"
sess = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider"])

input_name = sess.get_inputs()[0].name


In [ ]:
def preprocess(img_path, imgsz=640):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (imgsz, imgsz))
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))  # CHW
    return img[None]  # (1,3,H,W)


In [ ]:
import numpy as np

def parse_yolo_pose(out, kpt_num=21, conf_thres=0.25):
    """
    out: (1, 68, 8400)
    return: (21, 3) keypoints or None
    """
    out = out[0].T        # (8400, 68)

    scores = out[:, 4]    # objectness
    keep = scores > conf_thres
    if keep.sum() == 0:
        return None

    best = out[keep][scores[keep].argmax()]

    kpts = best[5:].reshape(kpt_num, 3)
    return kpts


In [ ]:
def infer_onnx(img_path):
    img = preprocess(img_path)  # (1,3,640,640)
    out = sess.run(None, {input_name: img})[0]
    return parse_yolo_pose(out)


In [ ]:
def oks(gt, pred, sigma=0.1):
    if gt is None or pred is None:
        return 0.0

    d = np.linalg.norm(gt[:, :2] - pred[:, :2], axis=1)
    oks = np.exp(-(d ** 2) / (2 * sigma ** 2))
    return oks.mean()


In [ ]:
dummy = sess.run(None, {input_name: preprocess(val_imgs[0])})
for i, o in enumerate(dummy):
    print(i, o.shape)


In [ ]:
print(gt.shape, pred.shape)


In [ ]:
kpts = infer_onnx(val_imgs[0])
print(kpts.shape)
print(kpts[:5])


In [ ]:
oks_scores = []

for img in tqdm(val_imgs):
    pred = infer_onnx(img)
    gt = load_gt(val_label_dir / f"{img.stem}.txt")
    oks_scores.append(oks(gt, pred))

np.mean(oks_scores)


In [ ]:
frame_t = kpts[:, :2]
frame_t1 = prev_kpts[:, :2]
jitter = np.linalg.norm(frame_t - frame_t1, axis=1).mean()


In [ ]:
def angle(a,b,c):
    ba = a - b
    bc = c - b
    return np.degrees(np.arccos(
        np.dot(ba,bc) / (np.linalg.norm(ba)*np.linalg.norm(bc)+1e-6)
    ))
